# Mapeamento Sistemático da Literatura — PRISMA 2020

Este notebook orquestra todo o workflow do mapeamento: carregamento, normalização, desduplicação, triagem, extração, qualidade e síntese.

In [22]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
import json
import os
from pathlib import Path

## 1. Criação de Diretórios

In [23]:
def create_directories(base_path="."):
    base = Path(base_path)
    dirs = [
        "dataset/raw",
        "dataset/processed",
        "dataset/extraction",
        "prisma",
        "outputs/tables",
        "outputs/figures",
        "outputs/summaries"
    ]
    for d in dirs:
        (base / d).mkdir(parents=True, exist_ok=True)
        
create_directories()
print("Diretórios criados/verificados com sucesso.")

Diretórios criados/verificados com sucesso.


## 2. Carregamento de Arquivos Brutos

In [24]:
def load_raw_files(base_path="."):
    base = Path(base_path)
    raw_dir = base / "dataset" / "raw"
    files = {
        "Scopus": raw_dir / "scopus_raw.csv",
        "Google Scholar": raw_dir / "google_scholar_raw.csv",
        "Semantic Scholar": raw_dir / "semantic_scholar_raw.csv",
        "IEEE": raw_dir / "IEEE_raw1.csv",
        "IEEE 2": raw_dir / "IEEE_raw2.csv",
        "OpenAlex": raw_dir / "openalex.csv"
    }
    
    dfs = []
    for source, path in files.items():
        if path.exists():
            try:
                df = pd.read_csv(path)
                if not df.empty:
                    # Se for base IEEE
                    if "IEEE" in source:
                        rename_map = {
                            "Document Title": "title",
                            "Authors": "authors",
                            "Publication Year": "year",
                            "Abstract": "abstract",
                            "DOI": "doi",
                            "PDF Link": "url",
                            "Publication Title": "venue"
                        }
                        df = df.rename(columns=rename_map)
                        
                    # Se for base OpenAlex
                    if "OpenAlex" in source:
                        rename_map = {
                            "display_name": "title",
                            "authorships.author.display_name": "authors",
                            "publication_year": "year",
                            "doi": "doi",
                            "id": "url",
                            "primary_location.source.display_name": "venue"
                        }
                        df = df.rename(columns=rename_map)
                    
                    # Unifica o nome da base
                    real_source = "IEEE" if "IEEE" in source else source
                    df['source_database'] = real_source
                    dfs.append(df)
            except Exception as e:
                print(f"Erro ao carregar {path}: {e}")
        else:
            print(f"Aviso: Arquivo {path} não encontrado.")
            
    if dfs:
        return pd.concat(dfs, ignore_index=True)
    return pd.DataFrame()

df_raw = load_raw_files()
print(f"Total de registros carregados: {len(df_raw)}")

Aviso: Arquivo dataset\raw\google_scholar_raw.csv não encontrado.
Aviso: Arquivo dataset\raw\semantic_scholar_raw.csv não encontrado.
Total de registros carregados: 3145


## 3. Normalização de Metadados

In [25]:
def normalize_metadata(df):
    if df.empty:
        return df

    expected_cols = ['id', 'title', 'authors', 'year', 'abstract', 'doi', 'url', 'source_database', 'venue', 'language', 'document_type']
    for col in expected_cols:
        if col not in df.columns:
            df[col] = np.nan

    if 'id' not in df.columns or df['id'].isna().all():
        df['id'] = range(1, len(df) + 1)

    if 'title' in df.columns:
        df['title'] = df['title'].astype(str).str.strip()

    if 'doi' in df.columns:
        df['doi'] = df['doi'].astype(str).str.strip().str.lower()
        df['doi'] = df['doi'].str.replace('https://doi.org/', '', regex=False)
        df.loc[df['doi'] == 'nan', 'doi'] = np.nan
        df.loc[df['doi'] == 'none', 'doi'] = np.nan

    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        
    df = df[expected_cols + [c for c in df.columns if c not in expected_cols]]
    return df

df_norm = normalize_metadata(df_raw)
print("Metadados normalizados.")

Metadados normalizados.


## 4. Desduplicação

In [26]:
def remove_duplicates(df, threshold=94):
    if df.empty:
        return df, pd.DataFrame(), {'duplicates_removed': 0}

    initial_count = len(df)
    df_original = df.copy()
    
    # 1. Exact DOI deduplication
    valid_doi_mask = df['doi'].notna() & (df['doi'] != '')
    df_with_doi = df[valid_doi_mask]
    df_no_doi = df[~valid_doi_mask]
    
    df_doi_dedup = df_with_doi.drop_duplicates(subset=['doi'], keep='first')
    df = pd.concat([df_doi_dedup, df_no_doi], ignore_index=True)
    
    # 2. Exact Title deduplication
    valid_title_mask = df['title'].notna() & (df['title'] != '')
    df_with_title = df[valid_title_mask]
    df_no_title = df[~valid_title_mask]
    
    df_title_dedup = df_with_title.drop_duplicates(subset=['title'], keep='first')
    df = pd.concat([df_title_dedup, df_no_title], ignore_index=True)
    
    # 3. Fuzzy Title deduplication
    titles = df['title'].dropna().tolist()
    ids_to_drop = set()
    title_to_id = dict(zip(df['title'], df['id']))
    
    titles_checked = set()
    for title in titles:
        if title in titles_checked:
            continue
        titles_checked.add(title)
        
        matches = process.extract(title, titles, scorer=fuzz.token_sort_ratio, limit=None)
        for match_title, score, match_index in matches:
            if score >= threshold and match_title != title:
                if match_title in title_to_id:
                    ids_to_drop.add(title_to_id[match_title])
                    titles_checked.add(match_title)
                    
    df_final = df[~df['id'].isin(ids_to_drop)]
    removed_ids = set(df_original['id']) - set(df_final['id'])
    
    final_count = len(df_final)
    report = {'duplicates_removed': initial_count - final_count}
    
    return df_final, pd.DataFrame({'removed_id': list(removed_ids)}), report

df_dedup, df_removed, dup_report = remove_duplicates(df_norm)

if not df_norm.empty:
    df_norm.to_csv("dataset/processed/literature_merged.csv", index=False)
    df_dedup.to_csv("dataset/processed/literature_dedup.csv", index=False)
    
print(f"Duplicatas removidas: {dup_report.get('duplicates_removed', 0)}. Total após desduplicação: {len(df_dedup)}")

Duplicatas removidas: 143. Total após desduplicação: 3002


## 5. Triagem por Título e Resumo

In [27]:
## 5. Triagem Automática por Título e Resumo

import re

# ============================================================
# Critérios baseados no README do projeto
# Tema: IA no ensino e na análise de performance de guitarra/violão
# ============================================================

INSTRUMENT_TERMS = [
    "guitar",
    "electric guitar",
    "classical guitar",
    "acoustic guitar",
    "guitarist",
    "guitarists",
    "guitar playing",
    "guitar performance",
    "guitar tablature",
    "tablature",
    "tabs",
    "fretboard",
    "chord",
    "chords",
    "finger placement",
    "violão",
    "guitarra",
    "guitarrista",
    "tablatura",
    "cifra",
    "acorde",
    "acordes"
]

MUSIC_EDUCATION_TERMS = [
    "music education",
    "music learning",
    "music teaching",
    "instrument learning",
    "instrumental learning",
    "music training",
    "musical training",
    "music instruction",
    "guitar learning",
    "guitar teaching",
    "guitar instruction",
    "guitar training",
    "student",
    "students",
    "learner",
    "learners",
    "teaching",
    "learning",
    "education",
    "pedagogy",
    "educational",
    "aprendizagem",
    "ensino",
    "educação",
    "estudante",
    "aluno",
    "alunos"
]

AI_TERMS = [
    "artificial intelligence",
    "ai",
    "machine learning",
    "deep learning",
    "neural network",
    "neural networks",
    "cnn",
    "convolutional neural network",
    "rnn",
    "recurrent neural network",
    "lstm",
    "gru",
    "transformer",
    "large language model",
    "llm",
    "computer vision",
    "pose estimation",
    "mediapipe",
    "reinforcement learning",
    "data mining",
    "intelligent system",
    "intelligent tutoring system",
    "adaptive system",
    "recommendation algorithm",
    "classification",
    "recognition",
    "prediction",
    "generative ai",
    "genetic algorithm",
    "inteligência artificial",
    "aprendizado de máquina",
    "aprendizado profundo",
    "rede neural",
    "redes neurais"
]

PERFORMANCE_ANALYSIS_TERMS = [
    "performance analysis",
    "performance evaluation",
    "performance assessment",
    "automatic assessment",
    "automatic evaluation",
    "feedback",
    "real-time feedback",
    "visual feedback",
    "personalized feedback",
    "assessment",
    "evaluation",
    "scoring",
    "score",
    "pitch",
    "rhythm",
    "timing",
    "tempo",
    "accuracy",
    "intonation",
    "technique",
    "playing technique",
    "chord recognition",
    "gesture recognition",
    "action recognition",
    "motion capture",
    "audio analysis",
    "audio signal",
    "tablature transcription",
    "automatic transcription",
    "music transcription",
    "performance tracking",
    "progress tracking",
    "adaptivity",
    "adaptive",
    "personalized learning",
    "personalization",
    "learning path",
    "analytics",
    "learning analytics",
    "avaliação",
    "feedback",
    "desempenho",
    "performance",
    "ritmo",
    "afinação",
    "técnica",
    "personalização",
    "adaptatividade"
]

# Termos que indicam forte chance de falso positivo.
# Eles só excluem diretamente quando não houver evidência forte de ensino/performance de guitarra.
EXCLUSION_TERMS = [
    "finance",
    "stock",
    "cryptocurrency",
    "corrosion",
    "steel",
    "microstructure",
    "manufacturing",
    "wooden blanks",
    "guitar manufacturing",
    "quality control",
    "landslide",
    "pulmonary tuberculosis",
    "covid",
    "hemodialysis",
    "stuttering",
    "intrusion detection",
    "cyberattack",
    "driving",
    "autonomous driving",
    "soil",
    "agriculture",
    "olive fruits",
    "tower asset",
    "telecommunication",
    "customer segmentation",
    "economics",
    "finance",
    "chemical",
    "schizophrenia",
    "seizure detection",
    "instagram",
    "tiktok",
    "point cloud",
    "object recognition",
    "image segmentation"
]


def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def find_terms(text, terms):
    text = normalize_text(text)
    found = []

    for term in terms:
        term_norm = normalize_text(term)

        # Termos curtos, como "ai", precisam aparecer como palavra isolada.
        if len(term_norm) <= 3:
            pattern = r"\b" + re.escape(term_norm) + r"\b"
            if re.search(pattern, text):
                found.append(term)
        else:
            if term_norm in text:
                found.append(term)

    return found


def classify_title_abstract(row):
    title = normalize_text(row.get("title", ""))
    abstract = normalize_text(row.get("abstract", ""))

    combined_text = f"{title} {abstract}"

    found_instrument = find_terms(combined_text, INSTRUMENT_TERMS)
    found_music_education = find_terms(combined_text, MUSIC_EDUCATION_TERMS)
    found_ai = find_terms(combined_text, AI_TERMS)
    found_performance = find_terms(combined_text, PERFORMANCE_ANALYSIS_TERMS)
    found_exclusion = find_terms(combined_text, EXCLUSION_TERMS)

    has_instrument = len(found_instrument) > 0
    has_music_education = len(found_music_education) > 0
    has_ai = len(found_ai) > 0
    has_performance = len(found_performance) > 0
    has_exclusion = len(found_exclusion) > 0

    # ============================================================
    # REGRA 1 — Inclusão forte
    # Guitarra/violão + IA + análise/feedback/avaliação/aprendizagem
    # ============================================================
    if has_instrument and has_ai and (has_performance or has_music_education):
        return pd.Series({
            "include_title_abstract": "sim",
            "exclusion_reason": "",
            "notes": (
                "Incluído automaticamente: estudo relacionado a guitarra/violão, IA e "
                "ensino, aprendizagem, feedback ou análise de performance. "
                f"Instrumento: {', '.join(found_instrument)}. "
                f"IA: {', '.join(found_ai)}. "
                f"Educação: {', '.join(found_music_education)}. "
                f"Performance/feedback: {', '.join(found_performance)}."
            )
        })

    # ============================================================
    # REGRA 2 — Inclusão possível
    # Guitarra/violão + IA, mas sem evidência clara de ensino/feedback
    # Pode ser útil para análise de performance, transcrição, tablatura ou datasets.
    # ============================================================
    if has_instrument and has_ai:
        return pd.Series({
            "include_title_abstract": "talvez",
            "exclusion_reason": "",
            "notes": (
                "Possivelmente relevante: há guitarra/violão e IA, mas o resumo não deixa claro "
                "o vínculo com ensino, feedback, avaliação ou aprendizagem. "
                f"Instrumento: {', '.join(found_instrument)}. "
                f"IA: {', '.join(found_ai)}."
            )
        })

    # ============================================================
    # REGRA 3 — Inclusão possível
    # Guitarra/violão + performance/feedback, mas IA não aparece claramente
    # ============================================================
    if has_instrument and has_performance:
        return pd.Series({
            "include_title_abstract": "talvez",
            "exclusion_reason": "",
            "notes": (
                "Possivelmente relevante: há guitarra/violão e análise de performance, feedback "
                "ou avaliação, mas a técnica de IA não aparece claramente no título/resumo. "
                f"Instrumento: {', '.join(found_instrument)}. "
                f"Performance/feedback: {', '.join(found_performance)}."
            )
        })

    # ============================================================
    # REGRA 4 — Estudos gerais de IA em educação musical
    # Não são diretamente sobre guitarra/violão, mas podem embasar RQs.
    # Mantém como talvez, não como sim.
    # ============================================================
    if has_music_education and has_ai and has_performance:
        return pd.Series({
            "include_title_abstract": "talvez",
            "exclusion_reason": "",
            "notes": (
                "Possivelmente relevante como estudo geral de IA em educação musical, mas sem "
                "evidência direta de guitarra elétrica ou violão. "
                f"Educação musical: {', '.join(found_music_education)}. "
                f"IA: {', '.join(found_ai)}. "
                f"Performance/feedback: {', '.join(found_performance)}."
            )
        })

    # ============================================================
    # REGRA 5 — Exclusão por falso positivo temático
    # ============================================================
    if has_exclusion and not has_instrument:
        return pd.Series({
            "include_title_abstract": "não",
            "exclusion_reason": "Fora do escopo temático: " + "; ".join(found_exclusion),
            "notes": "Excluído automaticamente por palavras-chave de falso positivo."
        })

    # ============================================================
    # REGRA 6 — Exclusão geral
    # ============================================================
    return pd.Series({
        "include_title_abstract": "não",
        "exclusion_reason": (
            "Não apresenta combinação mínima entre guitarra/violão, IA e ensino, "
            "feedback, avaliação ou análise de performance."
        ),
        "notes": "Excluído automaticamente por ausência dos critérios mínimos."
    })


def is_included(val):
    if pd.isna(val):
        return None

    val = str(val).strip().lower()

    if val in ["sim", "s", "yes", "y", "true", "1", "talvez", "maybe"]:
        return True

    if val in ["não", "nao", "n", "no", "false", "0"]:
        return False

    return None


def process_title_abstract_screening(df, output_path=None):
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    required_cols = ["title", "abstract"]

    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Coluna obrigatória ausente: {col}")

    screened_df = df.copy()

    if "include_title_abstract" not in screened_df.columns:
        screened_df["include_title_abstract"] = np.nan

    if "exclusion_reason" not in screened_df.columns:
        screened_df["exclusion_reason"] = np.nan

    if "notes" not in screened_df.columns:
        screened_df["notes"] = np.nan

    classification = screened_df.apply(classify_title_abstract, axis=1)

    screened_df["include_title_abstract"] = classification["include_title_abstract"]
    screened_df["exclusion_reason"] = classification["exclusion_reason"]
    screened_df["notes"] = classification["notes"]

    decisions = screened_df["include_title_abstract"].apply(is_included)

    included = screened_df[decisions == True].copy()
    excluded = screened_df[decisions == False].copy()
    pending = screened_df[decisions.isna()].copy()

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        screened_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    return screened_df, included, excluded, pending


ta_auto_path = Path("dataset/processed/title_abstract_screening_auto_classified.csv")

inc_ta = pd.DataFrame()
exc_ta = pd.DataFrame()
pending_ta = pd.DataFrame()

try:
    screened_ta, inc_ta, exc_ta, pending_ta = process_title_abstract_screening(
        df_dedup,
        output_path=ta_auto_path
    )

    inc_ta.to_csv(
        "dataset/processed/studies_included_after_screening.csv",
        index=False,
        encoding="utf-8-sig"
    )

    exc_ta.to_csv(
        "dataset/processed/studies_excluded_after_screening.csv",
        index=False,
        encoding="utf-8-sig"
    )

    pending_ta.to_csv(
        "dataset/processed/studies_pending_after_screening.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Triagem automática Título/Resumo processada: "
        f"{len(inc_ta)} incluídos, "
        f"{len(exc_ta)} excluídos, "
        f"{len(pending_ta)} pendentes."
    )

except Exception as e:
    print(f"Erro ao processar triagem automática: {e}")

Triagem automática Título/Resumo processada: 653 incluídos, 2349 excluídos, 0 pendentes.


## 6. Triagem por Texto Completo

In [ ]:
## 6. Triagem por Texto Completo / Recorte Temporal 2019–2026

def is_year_in_range(year, start=2019, end=2026):
    """
    Verifica se o ano do estudo está dentro do recorte temporal definido.
    Aceita valores como 2024, 2024.0 ou strings numéricas.
    """
    if pd.isna(year):
        return False

    try:
        year_int = int(float(year))
        return start <= year_int <= end
    except:
        return False


def generate_full_text_template(df, output_path):
    cols = [
        'id',
        'title',
        'authors',
        'year',
        'abstract',
        'source_database',
        'doi',
        'url',
        'include_title_abstract',
        'exclusion_reason',
        'notes'
    ]

    existing_cols = [c for c in cols if c in df.columns]
    template = df[existing_cols].copy()

    # Garante que year exista
    if 'year' not in template.columns:
        template['year'] = np.nan

    # Critério temporal
    template['year_in_range_2019_2026'] = template['year'].apply(is_year_in_range)

    # Cria colunas textuais como object/string, não como float
    template['full_text_available'] = ""
    template['include_full_text'] = ""
    template['exclusion_reason_full_text'] = ""
    template['full_text_status'] = ""
    template['full_text_notes'] = ""

    # Fora do recorte temporal
    out_of_range_mask = template['year_in_range_2019_2026'] == False

    template.loc[out_of_range_mask, 'full_text_available'] = 'não aplicável'
    template.loc[out_of_range_mask, 'include_full_text'] = 'não'
    template.loc[out_of_range_mask, 'exclusion_reason_full_text'] = (
        'Fora do recorte temporal definido no protocolo: 2019–2026.'
    )
    template.loc[out_of_range_mask, 'full_text_status'] = 'excluded_by_year'
    template.loc[out_of_range_mask, 'full_text_notes'] = (
        'Exclusão automática por critério temporal.'
    )

    # Dentro do recorte temporal
    in_range_mask = template['year_in_range_2019_2026'] == True

    template.loc[in_range_mask, 'full_text_status'] = 'pending_retrieval'
    template.loc[in_range_mask, 'full_text_notes'] = (
        'Texto completo ainda não avaliado.'
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    template.to_csv(output_path, index=False, encoding="utf-8-sig")


def is_full_text_unavailable(val):
    if pd.isna(val):
        return False

    val = str(val).strip().lower()

    unavailable_values = [
        'não',
        'nao',
        'n',
        'no',
        'false',
        '0',
        'indisponivel',
        'indisponível',
        'unavailable',
        'not available',
        'not retrieved'
    ]

    return val in unavailable_values


def process_full_text_screening(template_path):
    df = pd.read_csv(template_path)

    required_cols = [
        'year',
        'year_in_range_2019_2026',
        'full_text_available',
        'include_full_text',
        'exclusion_reason_full_text',
        'full_text_status',
        'full_text_notes'
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan

    # Recalcula o critério temporal, caso o CSV tenha sido editado manualmente
    df['year_in_range_2019_2026'] = df['year'].apply(is_year_in_range)

    # Exclui automaticamente estudos fora de 2019–2026
    out_of_range_mask = df['year_in_range_2019_2026'] == False

    df.loc[out_of_range_mask, 'include_full_text'] = 'não'
    df.loc[out_of_range_mask, 'exclusion_reason_full_text'] = 'Fora do recorte temporal definido no protocolo: 2019–2026.'
    df.loc[out_of_range_mask, 'full_text_status'] = 'excluded_by_year'
    df.loc[out_of_range_mask, 'full_text_notes'] = 'Exclusão automática por critério temporal.'

    decisions = df['include_full_text'].apply(is_included)
    unavailable = df['full_text_available'].apply(is_full_text_unavailable)

    included = df[
        (decisions == True) &
        (df['year_in_range_2019_2026'] == True)
    ].copy()

    excluded = df[
        (decisions == False) |
        (unavailable == True) |
        (df['year_in_range_2019_2026'] == False)
    ].copy()

    pending = df[
        decisions.isna() &
        (unavailable == False) &
        (df['year_in_range_2019_2026'] == True)
    ].copy()

    pending['full_text_status'] = 'pending_retrieval'
    pending['full_text_notes'] = 'Texto completo ainda não avaliado.'

    return included, excluded, pending, df


ft_template_path = Path("dataset/processed/full_text_screening_template.csv")

if not inc_ta.empty:
    if not ft_template_path.exists():
        generate_full_text_template(inc_ta, ft_template_path)
        print("Template de triagem por texto completo gerado com critério temporal 2019–2026.")
    else:
        print("Template de triagem por texto completo já existe.")

inc_ft = pd.DataFrame()
exc_ft = pd.DataFrame()
pending_ft = pd.DataFrame()

if ft_template_path.exists():
    try:
        inc_ft, exc_ft, pending_ft, full_text_screened_df = process_full_text_screening(ft_template_path)

        full_text_screened_df.to_csv(
            "dataset/processed/full_text_screening_auto_classified.csv",
            index=False,
            encoding="utf-8-sig"
        )

        inc_ft.to_csv(
            "dataset/processed/studies_included_final.csv",
            index=False,
            encoding="utf-8-sig"
        )

        exc_ft.to_csv(
            "dataset/processed/studies_excluded_full_text.csv",
            index=False,
            encoding="utf-8-sig"
        )

        pending_ft.to_csv(
            "dataset/processed/studies_pending_full_text.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Triagem Texto Completo processada: "
            f"{len(inc_ft)} incluídos finais, "
            f"{len(exc_ft)} excluídos, "
            f"{len(pending_ft)} pendentes de recuperação/leitura."
        )

    except Exception as e:
        print(f"Erro ao processar triagem full text: {e}")
else:
    print("Nenhum template de texto completo encontrado.")

TypeError: Invalid value 'não aplicável' for dtype 'float64'

## 7. Extração de Dados e Avaliação de Qualidade

In [ ]:
def generate_data_extraction_template(df, output_path):
    cols_from_df = ['id', 'title', 'authors', 'year']
    existing_cols = [c for c in cols_from_df if c in df.columns]
    template = df[existing_cols].copy()
    
    extraction_cols = [
        'venue', 'country', 'study_type', 'instrument', 'educational_context',
        'learner_level', 'ai_techniques', 'ml_dl_models', 'input_data_type',
        'audio_features', 'performance_dimensions', 'feedback_type',
        'assessment_method', 'adaptivity_personalization', 'progression_mechanism',
        'visualizations_analytics', 'evaluation_method', 'metrics', 'main_results',
        'limitations', 'future_work', 'related_research_questions'
    ]
    for col in extraction_cols:
        template[col] = np.nan
    template.to_csv(output_path, index=False)

def generate_quality_assessment_template(df, output_path):
    cols_from_df = ['id', 'title', 'authors', 'year']
    existing_cols = [c for c in cols_from_df if c in df.columns]
    template = df[existing_cols].copy()
    
    quality_cols = [
        'objective_clarity', 'methodological_adequacy', 'ai_technique_description',
        'dataset_quality', 'educational_context_clarity', 'feedback_description',
        'evaluation_presence', 'results_transparency', 'limitations_discussion',
        'replicability'
    ]
    for col in quality_cols:
        template[col] = np.nan
    template['total_score'] = np.nan
    template['quality_level'] = np.nan
    template['notes'] = np.nan
    template.to_csv(output_path, index=False)

data_ext_path = Path("dataset/extraction/data_extraction_template.csv")
qa_template_path = Path("dataset/extraction/quality_assessment_template.csv")

if not inc_ft.empty:
    if not data_ext_path.exists():
        generate_data_extraction_template(inc_ft, data_ext_path)
        print("Template de extração de dados gerado.")
    if not qa_template_path.exists():
        generate_quality_assessment_template(inc_ft, qa_template_path)
        print("Template de avaliação de qualidade gerado.")

## 8. Contagens PRISMA

In [ ]:
def calculate_prisma_counts(stats, output_path):
    required_keys = [
        "records_scopus", "records_google_scholar", "records_semantic_scholar", "records_ieee", "records_openalex",
        "records_total_identified", "duplicates_removed", "records_after_duplicates_removed",
        "records_screened_title_abstract", "records_excluded_title_abstract",
        "reports_sought_for_retrieval", "reports_not_retrieved",
        "reports_assessed_for_eligibility", "reports_excluded_full_text",
        "studies_included_final"
    ]
    
    stats["records_total_identified"] = stats.get("records_scopus", 0) + \
                                        stats.get("records_google_scholar", 0) + \
                                        stats.get("records_semantic_scholar", 0) + \
                                        stats.get("records_ieee", 0) + \
                                        stats.get("records_openalex", 0)
    
    stats["records_after_duplicates_removed"] = stats["records_total_identified"] - stats.get("duplicates_removed", 0)
    
    for key in required_keys:
        if key not in stats:
            stats[key] = 0
            
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=4)

stats = {}
if not df_raw.empty:
    stats['records_scopus'] = len(df_raw[df_raw['source_database'] == 'Scopus'])
    stats['records_google_scholar'] = len(df_raw[df_raw['source_database'] == 'Google Scholar'])
    stats['records_semantic_scholar'] = len(df_raw[df_raw['source_database'] == 'Semantic Scholar'])
    stats['records_ieee'] = len(df_raw[df_raw['source_database'] == 'IEEE'])
    stats['records_openalex'] = len(df_raw[df_raw['source_database'] == 'OpenAlex'])
    stats['duplicates_removed'] = dup_report.get('duplicates_removed', 0)
    stats['records_screened_title_abstract'] = len(df_dedup)
    stats['studies_included_after_screening'] = len(inc_ta)
    stats['records_excluded_title_abstract'] = len(exc_ta)
    stats['reports_sought_for_retrieval'] = len(inc_ta)
    stats['reports_assessed_for_eligibility'] = len(inc_ta)
    stats['studies_included_final'] = len(inc_ft)
    stats['reports_excluded_full_text'] = len(exc_ft)

calculate_prisma_counts(stats, "prisma/prisma_counts.json")
print("Contagens PRISMA calculadas e salvas.")

## 8.5 Geração de Gráficos e Visualizações

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurar estilo
sns.set_theme(style="whitegrid")
fig_dir = Path("outputs/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

# 1. Gráfico de Distribuição por Base de Dados (df_raw)
if not df_raw.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(data=df_raw, x='source_database', order=df_raw['source_database'].value_counts().index, palette='viridis')
    plt.title("Distribuição de Registros Identificados por Base de Dados", fontsize=14)
    plt.xlabel("Base de Dados", fontsize=12)
    plt.ylabel("Número de Registros", fontsize=12)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=11)
    plt.tight_layout()
    plt.savefig(fig_dir / "distribution_by_database.png", dpi=300)
    plt.close()

# 2. Gráfico de Publicações por Ano (df_dedup)
if not df_dedup.empty and 'year' in df_dedup.columns:
    year_counts = df_dedup['year'].value_counts().sort_index()
    # Filter valid years (e.g., 1900 to 2030)
    year_counts = year_counts[(year_counts.index >= 1900) & (year_counts.index <= 2030)]
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=year_counts.index.astype(int), y=year_counts.values, color='steelblue')
    plt.title("Evolução Temporal das Publicações (Após Desduplicação)", fontsize=14)
    plt.xlabel("Ano de Publicação", fontsize=12)
    plt.ylabel("Número de Publicações", fontsize=12)
    plt.xticks(rotation=45)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig(fig_dir / "publications_per_year.png", dpi=300)
    plt.close()

print("Gráficos gerados com sucesso na pasta outputs/figures.")

## 9. Geração dos Relatórios Finais (Markdown)

In [ ]:
def generate_results_md(stats, extracted_df, output_path):
    total = stats.get('records_total_identified', 0)
    scopus = stats.get('records_scopus', 0)
    scholar = stats.get('records_google_scholar', 0)
    semantic = stats.get('records_semantic_scholar', 0)
    ieee = stats.get('records_ieee', 0)
    openalex = stats.get('records_openalex', 0)
    duplicates = stats.get('duplicates_removed', 0)
    after_dedup = stats.get('records_after_duplicates_removed', 0)
    inc_screen = stats.get('studies_included_after_screening', 0)
    inc_final = stats.get('studies_included_final', 0)
    
    # Analyze extracted data if available
    ai_tech_summary = "Nenhum estudo final incluído encontrado."
    input_data_summary = "Aguardando preenchimento da extração de dados."
    feedback_summary = "Aguardando preenchimento da extração de dados."
    pedagogy_summary = "Aguardando preenchimento da extração de dados."
    
    if not extracted_df.empty:
        if 'ai_techniques' in extracted_df.columns:
            techs = extracted_df['ai_techniques'].dropna().value_counts().to_string()
            ai_tech_summary = f"Técnicas mais comuns identificadas nos dados extraídos:\n\n```text\n{techs}\n```"
        else:
            ai_tech_summary = "Coluna 'ai_techniques' não encontrada nos dados."
            
        if 'input_data_type' in extracted_df.columns:
            inputs = extracted_df['input_data_type'].dropna().value_counts().to_string()
            input_data_summary = f"Tipos de dados de entrada mais comuns:\n\n```text\n{inputs}\n```"
        else:
            input_data_summary = "Coluna 'input_data_type' não encontrada nos dados."
            
        if 'feedback_type' in extracted_df.columns:
            feedbacks = extracted_df['feedback_type'].dropna().value_counts().to_string()
            feedback_summary = f"Tipos de feedback mais comuns:\n\n```text\n{feedbacks}\n```"
        else:
            feedback_summary = "Coluna 'feedback_type' não encontrada nos dados."
            
        if 'educational_context' in extracted_df.columns:
            contexts = extracted_df['educational_context'].dropna().value_counts().to_string()
            pedagogy_summary = f"Contextos educacionais mais comuns:\n\n```text\n{contexts}\n```"
        else:
            pedagogy_summary = "Coluna 'educational_context' não encontrada nos dados."
    
    content = f"""# Resultados do Mapeamento Sistemático

## 1. Visão Geral dos Registros

O fluxograma PRISMA baseia-se nos seguintes quantitativos:
- **Total de registros identificados:** {total}
  - Scopus: {scopus}
  - Google Scholar: {scholar}
  - Semantic Scholar: {semantic}
  - IEEE Xplore: {ieee}
  - OpenAlex: {openalex}
- **Duplicatas removidas:** {duplicates}
- **Registros após desduplicação:** {after_dedup}
- **Estudos incluídos após triagem (Título e Resumo):** {inc_screen}
- **Estudos incluídos após triagem (Texto Completo):** {inc_final}

### 1.1 Distribuição nas Bases de Dados

![Distribuição de Registros](outputs/figures/distribution_by_database.png)

### 1.2 Evolução Temporal das Publicações

![Evolução Temporal](outputs/figures/publications_per_year.png)

## 2. Caracterização dos Estudos Incluídos

{"Os dados ainda não foram extraídos. Proceda ao preenchimento do template `dataset/extraction/data_extraction_template.csv` e salve como `data_extraction_completed.csv`." if extracted_df.empty else f"Foram extraídos dados de {len(extracted_df)} estudos finais."}

## 3. Técnicas de Inteligência Artificial Identificadas

{ai_tech_summary}

## 4. Dados e Sinais de Entrada Utilizados

{input_data_summary}

## 5. Formas de Feedback e Avaliação de Performance

{feedback_summary}

## 6. Aspectos Pedagógicos e Adaptatividade

{pedagogy_summary}

## 7. Relação com as Questões de Pesquisa (RQ)

Aqui apresentamos uma síntese direta das métricas extraídas para as questões do protocolo:

- **RQ1:** Mapeamento inicial das métricas.
- **RQ2:** Identificação tecnológica via extração.
- **RQ3 e RQ4:** Consolidação em andamento com base nos estudos selecionados.
"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(content)

def generate_conclusions_md(extracted_df, output_path):
    content = f"""# Conclusões do Mapeamento Sistemático

## 1. Síntese Geral

{"Conclusões preliminares, pois a extração de dados ainda está incompleta ou vazia." if extracted_df.empty else "Apresentar uma síntese geral sobre o estado da literatura com base nos dados extraídos."}

## 2. Principais Achados

- tipos de sistemas encontrados;
- técnicas de IA utilizadas;
- dados de entrada empregados;
- formas de feedback;
- avaliação de performance;
- adaptatividade;
- lacunas técnicas;
- lacunas pedagógicas.

## 3. Implicações para o Desenvolvimento de Sistemas de Aprendizagem de Guitarra/Violão

Discutir como os resultados podem orientar o desenvolvimento de sistemas mais dinâmicos, adaptativos e pedagogicamente fundamentados.

## 4. Limitações do Mapeamento

Limitações relacionadas a:
- bases consultadas;
- string de busca;
- critérios de seleção;
- disponibilidade de texto completo;
- heterogeneidade dos estudos;
- dependência de decisões manuais de triagem e extração.

## 5. Trabalhos Futuros

Caminhos futuros incluem:
- desenvolvimento de ambientes adaptativos para aprendizagem de guitarra;
- integração de feedback em tempo real;
- uso de análise automática de áudio;
- suporte a tablatura e afinações alternativas;
- uso de dashboards de aprendizagem;
- avaliação empírica com estudantes;
- criação de datasets públicos para performance de guitarra/violão.

## 6. Consideração Final

A Inteligência Artificial atua como mediadora pedagógica potencial para o ensino instrumental, ampliando possibilidades de feedback, personalização e acompanhamento do estudante sem substituir o professor.
"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(content)

ext_path = Path("dataset/extraction/data_extraction_completed.csv")
if ext_path.exists():
    try:
        ext_df = pd.read_csv(ext_path)
    except:
        ext_df = pd.DataFrame()
else:
    ext_df = pd.DataFrame()

generate_results_md(stats, ext_df, "resultados.md")
generate_conclusions_md(ext_df, "conclusoes.md")
print("Relatórios gerados com sucesso (resultados.md e conclusoes.md). Os gráficos foram incorporados em resultados.md.")